# Exploring BigQuery

- BigQuery metadata, i.e. information about a table
- BigQuery query limits with QueryJobConfig()


## Setup

In [1]:
from google.cloud import bigquery
from google.colab import auth
import pandas as pd
from google.colab import userdata


In [2]:
auth.authenticate_user()

In [3]:
billing_project_id=userdata.get('bq_billing_project_id')
billing_project_id


'my-ddds-project'

In [4]:
# Create client object
client = bigquery.Client(project=billing_project_id)


## Table Metadata Exploration

### List the tables

In [5]:
# Construct a reference to the "Global Biodiversity Information Facility" dataset
dataset_ref = client.dataset("gbif", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

# Get all the tables in the dataset
tables = list(client.list_tables(dataset))

# Print names of all tables in the dataset
for table in tables:
  print(table.table_id)

occurrences


In [6]:
table_id = "occurrences"
table_id

'occurrences'

### Look at the table schema

In [7]:
# Construct a reference to the "mobility report" table
table_ref = dataset.table("occurrences")

# API request - fetch the table
table = client.get_table(table_ref)

# See the table's schema - name, field type, mode, description
table.schema

[SchemaField('gbifid', 'STRING', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('datasetkey', 'STRING', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('occurrenceid', 'STRING', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('kingdom', 'STRING', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('phylum', 'STRING', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('class', 'STRING', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('order', 'STRING', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('family', 'STRING', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('genus', 'STRING', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('species', 'STRING', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('infraspecificepithet', 'STRING', 'NULLABLE', None, None, (), None, None, None, None),
 SchemaField('taxonrank', 'STRI

In [8]:
# convert the table.schema into a data frame
fields = pd.DataFrame( [ x.to_api_repr() for x in table.schema ] )
fields.head()


,name,type,mode,fields
0,gbifid,STRING,NULLABLE,NaN
1,datasetkey,STRING,NULLABLE,NaN
2,occurrenceid,STRING,NULLABLE,NaN
3,kingdom,STRING,NULLABLE,NaN
4,phylum,STRING,NULLABLE,NaN


In [9]:
fields

,name,type,mode,fields
0,gbifid,STRING,NULLABLE,NaN
1,datasetkey,STRING,NULLABLE,NaN
2,occurrenceid,STRING,NULLABLE,NaN
3,kingdom,STRING,NULLABLE,NaN
4,phylum,STRING,NULLABLE,NaN
5,class,STRING,NULLABLE,NaN
6,order,STRING,NULLABLE,NaN
7,family,STRING,NULLABLE,NaN
8,genus,STRING,NULLABLE,NaN
9,species,STRING,NULLABLE,NaN


In [10]:
fields.shape

(50, 4)

In [11]:
# Preview the first five lines of the table as a data frame
client.list_rows(table, max_results=5).to_dataframe().transpose()


,0,1,2,3,4
gbifid,1836917927,1843227470,1843227863,3393317370,3393318372
datasetkey,1e61b812-b2ec-43d0-bdbb-8534a761f74c,1e61b812-b2ec-43d0-bdbb-8534a761f74c,1e61b812-b2ec-43d0-bdbb-8534a761f74c,1e61b812-b2ec-43d0-bdbb-8534a761f74c,1e61b812-b2ec-43d0-bdbb-8534a761f74c
occurrenceid,fb940509-2914-9144-8894-efa5bef10a21,b622f9d5-e282-2824-eda8-c3098f49bfd9,c3722904-1d08-d084-3790-6c4bb428e188,c6ed12f6-cec6-ec64-e684-8a47e8f97509,b5c0824c-1f95-f954-0f9a-090146137e74
kingdom,None,None,None,None,None
phylum,None,None,None,None,None
class,None,None,None,None,None
order,None,None,None,None,None
family,None,None,None,None,None
genus,None,None,None,None,None
species,None,None,None,None,None


##  Add safe config settings

BigQuery allows you to query up to 1 TB per month. You can quickly reach this limit if you are not careful. Luckily, there are ways to assess and limit the amount of data you are querying.

Set constants for sizes

In [12]:
ONE_MB = 1_000*1_000
ONE_GB = 1_000*ONE_MB

### Sample Queries - Dry Run

You can use a 'dry run' to estimate the size of a query before running it.

In [13]:
project_id = "bigquery-public-data"
dataset_id = "gbif"
table_id = "occurrences"

queries = []

queries += [ f"""
        SELECT countrycode
        FROM {project_id}.{dataset_id}.{table_id}
        WHERE class = "Magnoliopsida" AND countrycode ='US'
        LIMIT 10
        """ ]

queries += [ f"""
        SELECT countrycode
        FROM {project_id}.{dataset_id}.{table_id}
        WHERE class = "Magnoliopsida" AND countrycode ='US'
        """ ]

queries += [ f"""
        SELECT countrycode
        FROM {project_id}.{dataset_id}.{table_id}
        WHERE countrycode ='US'
        """ ]

queries += [ f"""
        SELECT year
        FROM {project_id}.{dataset_id}.{table_id}
        WHERE year = 2012
        """ ]

queries += [ f"""
        SELECT countrycode, class
        FROM {project_id}.{dataset_id}.{table_id}
        WHERE class = "Magnoliopsida" AND countrycode ='US'
        """ ]

queries += [ f"""
        SELECT countrycode, class
        FROM {project_id}.{dataset_id}.{table_id}
        WHERE countrycode ='US'
        """ ]

queries += [ f"""
        SELECT countrycode, class
        FROM {project_id}.{dataset_id}.{table_id}
        """ ]

queries += [ f"""
        SELECT countrycode
        FROM {project_id}.{dataset_id}.{table_id}
        """ ]

queries += [ f"""
        SELECT count(countrycode)
        FROM {project_id}.{dataset_id}.{table_id}
        """ ]

queries += [ f"""
        SELECT count(1)
        FROM {project_id}.{dataset_id}.{table_id}
        """ ]

queries += [ f"""
        SELECT count(countrycode)
        FROM {project_id}.{dataset_id}.{table_id}
        WHERE countrycode ='US'
        """ ]

queries += [ f"""
        SELECT count(1)
        FROM {project_id}.{dataset_id}.{table_id}
        WHERE countrycode ='US'
        """ ]

queries += [ f"""
        SELECT count(*)
        FROM {project_id}.{dataset_id}.{table_id}
        """ ]

queries += [ f"""
        SELECT *
        FROM {project_id}.{dataset_id}.{table_id}
        LIMIT 1
        """ ]
len(queries)


14

In [14]:
for query in queries:
  dry_run_config = bigquery.QueryJobConfig(dry_run = True)
  dry_run_query_job = client.query(query, job_config= dry_run_config)
  size = dry_run_query_job.total_bytes_processed
  print(query)
  print(f"{size:_}")
  print()


        SELECT countrycode
        FROM bigquery-public-data.gbif.occurrences
        WHERE class = "Magnoliopsida" AND countrycode ='US'
        LIMIT 10
        
49_476_099_411


        SELECT countrycode
        FROM bigquery-public-data.gbif.occurrences
        WHERE class = "Magnoliopsida" AND countrycode ='US'
        
49_476_099_411


        SELECT countrycode
        FROM bigquery-public-data.gbif.occurrences
        WHERE countrycode ='US'
        
15_598_074_688


        SELECT year
        FROM bigquery-public-data.gbif.occurrences
        WHERE year = 2012
        
30_223_092_896


        SELECT countrycode, class
        FROM bigquery-public-data.gbif.occurrences
        WHERE class = "Magnoliopsida" AND countrycode ='US'
        
49_476_099_411


        SELECT countrycode, class
        FROM bigquery-public-data.gbif.occurrences
        WHERE countrycode ='US'
        
49_476_099_411


        SELECT countrycode, class
        FROM bigquery-public-data.gbif.occurren

Sample Query 1 - Safe Config
You can also specify a limit for how much data you want to scan.

In [15]:
# Create a list of queries
queries = []

queries += [ f"""
        SELECT count(1) as `total`
        FROM {project_id}.{dataset_id}.{table_id}
        """ ]

queries += [ f"""
        SELECT count(1) as `total`
        FROM {project_id}.{dataset_id}.{table_id}
        WHERE countrycode ='US'
        """ ]

queries += [ f"""
        SELECT count(countrycode) as `total`
        FROM {project_id}.{dataset_id}.{table_id}
        """ ]

queries += [ f"""
        SELECT count(countrycode) as `total`
        FROM {project_id}.{dataset_id}.{table_id}
        WHERE countrycode ='US'
        """ ]

len(queries)

4

In [23]:
# safe_config needs to be included with every client.query() request
safe_config = bigquery.QueryJobConfig(
    # maximum_bytes_billed=ONE_GB*100,
    # maximum_bytes_billed=ONE_GB,
    # maximum_bytes_billed=ONE_MB,
    maximum_bytes_billed=0,
    # totalBytesProcessed=ONE_GB,
    # total_bytes_processed=ONE_GB,
)
# Use a try...except block to catch when the safe_config paramenter prevents a query
for query in queries:
  print(query)
  try:
    df = client.query(query, job_config=safe_config).to_dataframe()
    display(df.head())
  except:
    print("Blocked by safe_config")



        SELECT count(1) as `total`
        FROM bigquery-public-data.gbif.occurrences
        


,total
0,3954221790



        SELECT count(1) as `total`
        FROM bigquery-public-data.gbif.occurrences
        WHERE countrycode ='US'
        


,total
0,1321084889



        SELECT count(countrycode) as `total`
        FROM bigquery-public-data.gbif.occurrences
        


,total
0,3899518672



        SELECT count(countrycode) as `total`
        FROM bigquery-public-data.gbif.occurrences
        WHERE countrycode ='US'
        


,total
0,1321084889


In [24]:
f"{ONE_GB:_}"

'1_000_000_000'

In [25]:
2669694827 > ONE_GB

True